In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
from mtrain.utils import mkdir
import json
from PIL import Image
from pycocotools import mask as mask_utils
from tqdm import tqdm
from mtrain.disk import DiskBooleanMask, DiskImage
import itertools

In [ ]:
DS = Path("../../datasets/")
BASE = DS / "test-samples"
NEG_MASKING_V1 = BASE / "neg-masking" / "V1"
TRASH = NEG_MASKING_V1 / "trash"
SAMPLES_MAPILLARY = NEG_MASKING_V1 / "samples_mapillary"
MODEL_DIR = DS / "models" / "trash_classification"

CLIP_FILE_NAMES = [
    "clip_bottles.txt",
    "clip_litter.txt",
    "clip_plastic.txt",
    "clip_tobacco_packs.txt",
    "clip_delhi_litter.txt",
]
CLIP_FILES = [TRASH / c for c in CLIP_FILE_NAMES]
print("all clip files exist:", all([f.exists() for f in CLIP_FILES]))

TRASH_DATA_DIR = TRASH / "data"

SAMPLES_MAPILLARY.exists(), TRASH.exists()

In [ ]:
from mtrain.neg_mask.ipywidgets.widget_2 import LabelWidget
from mtrain.neg_mask.crops import get_region_crops, Bbox

In [ ]:
def read_clip_file(path) -> list[tuple[str, Path]]:
    with open(path) as f:
        lines = f.readlines()
    imgs = [Path(line.split("\t")[1].strip()) for line in lines]
    dirs = [(path.stem, img.parent) for img in imgs]
    return dirs


def get_all_from_dir(path) -> list[tuple[str, Path]]:
    return [(path.stem, d) for d in path.glob("*")]


def get_dir_and_cat_iter():
    all_cat_dirs = [read_clip_file(f) for f in CLIP_FILES]
    # all_cat_dirs.append(get_all_from_dir(TRASH / "personal"))
    return itertools.chain.from_iterable(itertools.zip_longest(*all_cat_dirs))


def get_dir_for_widget():
    return (d for (_, d) in get_dir_and_cat_iter())

# Classfication dataset

In [ ]:
LABELS = ["other", "trash"]

In [ ]:
# it = get_dir_for_widget()

it = iter(get_all_from_dir(TRASH / "personal"))

In [ ]:
plt.imshow(plt.imread(next(it)[1]/ "image.jpg"))

In [ ]:
from torchvision.models import mobilenet_v3_small
from mtrain.neg_mask.ipywidgets.widget_3 import EvalWidget
from mtrain.neg_mask.model.learner import load_our_learner, dummy_dls
from mtrain.neg_mask.openai_clip import interleaved_data_from_multiple_clip_files
from fastai.vision.all import (
    mobilenet_v3_large,
    vision_learner,
    accuracy,
    F1Score,
    CrossEntropyLossFlat,
    ProgressCallback,
    default_device,
)

learner = vision_learner(
    dummy_dls(LABELS),
    mobilenet_v3_small,
    n_in=12,
    metrics=[accuracy, F1Score(average="macro")],
    loss_func=CrossEntropyLossFlat(),
    n_out=len(LABELS),
    normalize=False,
)
learner = learner.remove_cb(ProgressCallback)

learner = learner.load("/Users/hariomnarang/Desktop/personal/roads/datasets/models/trash_classification/mobilenet_v3_small-chan_12-iter_11")

# learn = load_our_learner(dls, mobilenet_v3_large, None, LABELS)
# state_dict = torch.load('/Users/hariomnarang/Desktop/personal/roads/datasets/models/taco_pretrained_mask_classifier/mobilenet_v3_large_130x130_iter-20-pure-torch.pth')
# keys_to_remove = [k for k in state_dict.keys() if '1.8' in k]
# print("Removing:", keys_to_remove)
# for k in keys_to_remove:
#     del state_dict[k]

# learn.model.load_state_dict(state_dict, strict=False)

In [ ]:
it = iter(get_all_from_dir(Path("/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/personal")))
it = (d for (n,d) in it)

In [ ]:
ROCKS = NEG_MASKING_V1 / "rocks"
# classification folder is here
# widget = LabelWidget(ROCKS / "classification", crop_pad=220, learner=learner)
widget = LabelWidget(ROCKS / "classification", crop_pad=220)


def is_dir_not_done(direc: Path):
    return not widget.is_done(direc.name)
it = filter(is_dir_not_done, it)


dir_it = filter(is_dir_not_done, iter(get_dir_for_widget()))

# run the cell below for vscode color theme support in ipywidgets

# Source - https://stackoverflow.com/a/77028015
# Posted by Yingding Wang, modified by community. See post 'Timeline' for change history
# Retrieved 2026-03-02, License - CC BY-SA 4.0

In [ ]:
d = next(it)
img, mask = DiskImage.load(d / "image.jpg"), DiskBooleanMask.load(d / "m2.png")
bboxes = list(get_region_crops(img, mask))

widget.ui(d, bboxes, img, mask)

# Model performance analysis

In [ ]:
from mtrain.neg_mask.ipywidgets.widget_3 import EvalWidget
from mtrain.neg_mask.model.learner import load_our_learner, dummy_dls
from mtrain.neg_mask.openai_clip import interleaved_data_from_multiple_clip_files
from fastai.vision.all import mobilenet_v3_large

model = (MODEL_DIR / "mobilenet_large_mask_thres_6_25_epochs").resolve()
learner = load_our_learner(dummy_dls(), mobilenet_v3_large, None, model)

In [ ]:
import itertools


def widget_iter(files):
    for name, image_dir in interleaved_data_from_multiple_clip_files(files):
        yield (image_dir / "image.jpg").resolve(), (image_dir / "m2.png").resolve()


iterator = widget_iter(CLIP_FILES)

In [ ]:
widget = EvalWidget(learner, iterator)

In [ ]:
# generate the masks first
from fastai.vision.all import load_learner

learner100 = load_learner(
    "/Users/hariomnarang/Desktop/gdrive-sync/garbage/experiments/enguled-bbox-levels-crops-v3/log/export_iter_14.pkl"
)
SIZE = 100

In [ ]:
from mtrain.smallnet.unet.predict.strided import single
from mtrain.disk import DiskImage, DiskBooleanMask
from tqdm import tqdm

img_dirs = list((TRASH / "delhi_litter").glob("*"))
for d in tqdm(img_dirs):
    img = DiskImage.load(d / "image.jpg")
    mask = single.strided_predict_unet_only_mask(img, 100, learner100, [50])
    DiskBooleanMask.save(mask, "m2.png")

In [ ]:
from mtrain.neg_mask.openai_clip import (
    get_images_from_clip_file,
    get_image_dirs_from_clip_file,
)

for _, d in tqdm(get_image_dirs_from_clip_file(TRASH / "clip_litter.txt")[:100]):
    img = DiskImage.load(d / "image.jpg")
    mask = single.strided_predict_unet_only_mask(img, 100, learner100, [50])
    DiskBooleanMask.save(mask, "m2.png")